# AWS Lambda Function to Process CSV and Generate Plot
This notebook contains a Lambda function that reads a CSV file from S3, generates a graph, and saves it back to S3.

In [ ]:
import json
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import io

s3 = boto3.client('s3')

def lambda_handler(event, context):
    bucket_name = "acceleration-research-s3-bucket"
    input_key = "SDL1/Input/sample_data.csv"
    output_key = "SDL1/Output/sample_plot.png"
    
    # Download CSV from S3
    response = s3.get_object(Bucket=bucket_name, Key=input_key)
    df = pd.read_csv(io.BytesIO(response['Body'].read()))
    
    # Convert 'Date' column to datetime
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Plot the data
    plt.figure(figsize=(8, 5))
    plt.plot(df['Date'], df['Value'], marker='o', linestyle='-')
    plt.xlabel("Date")
    plt.ylabel("Value")
    plt.title("Sample Data Plot")
    plt.grid(True)

    # Save the plot to a BytesIO buffer
    img_buffer = io.BytesIO()
    plt.savefig(img_buffer, format="png")
    img_buffer.seek(0)

    # Upload the plot to S3
    s3.put_object(Bucket=bucket_name, Key=output_key, Body=img_buffer, ContentType="image/png")
    
    return {
        "statusCode": 200,
        "body": json.dumps(f"Plot saved to s3://{bucket_name}/{output_key}")
    }